In [1]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import nnx
import nnx.functional as Fx
import nnx.graphics as G
import nnx.projection as P
import nnx.console as console

/home/wipkat/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
# Colors

console.print("Foreground colors can be set by <pink>name</>")
console.print("or by <#480>short hexstring</> or <#F86AB4>long hexstring</>")
console.print("You can also make text <b>bold</b> or <u>underlined</u> etc.")
console.print()
console.print("<u>These are the predefined controls:</>")
for name, (value, description) in console.color.registered_colors.items():
    dname = f"<{name}>"
    console.print(f"{dname:12s} = [{name}]{description}[/]", control="[]")
console.print()
console.print("The background can be set by <~blue>prepending <~831>(or replacing # with)<~blue> a tilde (~)</>")
console.print("or you can set both the <lime~8800FF>background and foreground at once</>")

Foreground colors can be set by name 
or by short hexstring or long hexstring 
You can also make text bold or underlined etc. 

These are the predefined controls: 
<red>        = color red (#FF0000) 
<green>      = color green (#00FF00) 
<blue>       = color blue (#0000FF) 
<yellow>     = color yellow (#FFFF00) 
<orange>     = color orange (#FF8800) 
<lime>       = color lime (#88FF00) 
<cyan>       = color cyan (#00FFFF) 
<teal>       = color teal (#00FF88) 
<magenta>    = color magenta (#FF00FF) 
<pink>       = color pink (#FF4488) 
<purple>     = color purple (#8800FF) 
<white>      = color white (#FFFFFF) 
<lightgray>  = color lightgray (#BBBBBB) 
<gray>       = color gray (#777777) 
<darkgray>   = color darkgray (#222222) 
<black>      = color black (#000000) 
</>          = reset to default 
<b>          = bold 
</b>         = not bold 
<f>          = f? 
</f>         = not f? 
<i>          = italic 
</i>         = not italic 
<u>          = underlined 
</u>         = not underli

In [3]:
# Wrapped text
text = "This is a <b>long</b> string with <pink>some control characters and <lime~darkgray>color</>"
progress = console.ProgressBar(100, title="Progress")
to_wrap = text + " " + console.style_str(progress)
console.print(to_wrap)
print()

wrapped = console.wrap(to_wrap, 18)
print(wrapped)
print()

console.print(wrapped)
print()

# padded text
padded = console.pad(wrapped, 1, pad_char=".")
console.print(padded)
print()

padded = console.pad(wrapped, 1, pad_char=".", align="center")
console.print(padded)
print()

# still some error for progress somewhere... checking later


This is a long string with some control characters and color Progress  ■■■■■■■■■■  0.0% 

This is a <b>long</b>
string with <pink>some
control characters
and <lime~darkgray>color</> Progress
 ■■■■■■■■■■  0.0%

This is a long
string with some
control characters
and color Progress
 ■■■■■■■■■■  0.0% 

....................
.This is a long.....
.string with some...
.control characters.
.and color Progress.
. ■■■■■■■■■■  0.0%..
.................... 

....................
...This is a long...
..string with some..
.control characters.
.and color Progress.
. ■■■■■■■■■■  0.0%..
.................... 



In [4]:
# Setting up something like a meter

timer = console.Timer()
progress = console.ProgressBar(100, title="Progress")
value1 = console.EMA(gamma=0.99, title="A:<pink>")
value2 = console.EMA(gamma=0.9, title="B:<orange>")

for i in range(progress.min_value, progress.max_value):
    progress.value = i+1
    value1.update(torch.randn(()))
    value2.update(torch.randn(()))
    console.print(timer, progress, value1, value2, end="   \r")
    time.sleep(0.1)

[00:00:09] Progress  ■■■■■■■■■■  100.0% A: 0.0707 (+-0.9774) B: -0.0449 (+-1.4724)    

In [5]:
# How to use console in conjunction with standard logging as 
# plain text.

class OnlyStyledForConsole:
    def __style_str__(self, style=None):
        return "<red>__style_str__ is used first by console if available</>"
    def __str__(self):
        return "otherwise __str__ is used"

class Other:
    def __str__(self):
        return "always returns plain string"
        
a = OnlyStyledForConsole()
b = Other()

console.print(a)
console.print(b)
print()
print(a)
print(b)

__style_str__ is used first by console if available 
always returns plain string 

otherwise __str__ is used
always returns plain string


In [6]:
### TODO, make __style_str__ for timer, EMA and progress